### Credit Card Fraud Detection (EDA)

Dataset: 284,807 credit card transactions, 492 frauds (~0.17%).

Target: Class (1 = fraud, 0 = legit).


Normal transactions (Class = 0): ~99.83%<br>
Fraud transactions (Class = 1):  ~0.17%

**Important note:** The dataset is extremely imbalanced, with fraud transactions representing only ~0.17% of all observations. <br>A model could easily achieve 99.8% accuracy by predicting almost everything as legitimate and still perform poorly at detecting fraud. <br>We will use PR-AUC instead. We'll also track Precision, Recall, and F1-score to get a more complete picture of model performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
data = pd.read_csv('../data/creditcard.csv')
data.head()

In [ ]:
print(data.Time.min(), data.Time.max())

In [ ]:
data[['V1','V2','V3']].describe()

In [ ]:
data[[f'V{i}' for i in range(1, 29)]].describe().T

In [ ]:
print(f"\nMissing values: {data.isnull().sum().sum()}")
print(f"Memory: {data.memory_usage().sum() / 1024**2:.1f} MB")

In [ ]:
print("Count: ", data.Class.value_counts())
print("\nProcent: ", data.Class.value_counts(normalize=True)*100)

In [ ]:
# Comparing Fraud transactions with Normal transactions
print(np.mean(data.Amount[data.Class == 0]))
print(np.mean(data.Amount[data.Class == 1]))

In [ ]:
print("Normal transactions: \n", data.Amount[data.Class == 0].describe())
print("\nFraud transactions: \n", data.Amount[data.Class == 1].describe())

Median of fraud transactions is lower compared to legit transactions.

Mean is higher for frauds, but this is misleading. It's pulled up by a few outliers. 
The median tells the real story: typical fraud is smaller than typical legit transaction.

Interesting note: 25% frauds are very small amounts (less than $1)

Many small frauds might suggest something like card testing, to verify stolen cards with micro-payments, before larger transactions.

In [ ]:
hours = data['Time'] / 3600

#fig, (ax1, ax2) = plt.subplots(1, 2)

#ax1.hist(hours[data.Class == 0], bins=30)
#ax1.set_title('Real Transactions')

#ax2.hist(hours[data.Class == 1], bins=30)
#ax2.set_title('Fraud Transactions')

fig, ax = plt.subplots(figsize=(12,6))
ax.hist(hours[data.Class == 0], bins=30, density=True, alpha=0.5, label='Legit')
ax.hist(hours[data.Class == 1], bins=30, density=True, alpha=0.5, label='Fraud')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3) 
ax.set_title('Fraud vs Real transactions')
ax.set_xlabel('Hours from first transaction')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.show()


X-axis shows Hours from first transaction (we don't see exact hours, all we know we have data from 48 hours)
Y-axis shows density (proportion of transactions per hour)

Legit transactions show a clear pattern repeating every ~24h (two cycles visible in 48h dataset). 
This likely shows day/night human activity, with peaks during days and dips during night.

Fraud transactions show different pattern, with peaks during low-activity periods.

In [ ]:
corr_matrix = data.corr()["Class"].abs().sort_values(ascending=False)
corr_matrix

To get a better idea of which features might be useful for fraud detection, I looked at their correlation with the target variable (`Class`)
Below are the features with the strongest absolute Pearson correlation values.

**Top correlation signals:**
- V17: 0.33
- V14: 0.30
- V12: 0.26
- V10: 0.22
- V16: 0.20
- V3:  0.19
- V7:  0.19
- V11: 0.15

These features seem to have the strongest relationship with fraud. Correlation only captures linear relationships, so features with lower correlation may still be important for more complex models.

## Key takeaways

1. **Huge Class imbalance:** Fraud transactions are just ~0.17% of all transactions, it means accuracy will be useless here to determine model quality
2. **Fraud amounts profile:** Typical fraud has lower amount than normal transactions, 25% of frauds below $1 may suggest card testing pattern
3. **Time patterns:** Legit transactions follow a clear daily cycle; fraud peaks in low-activity windows. Suggests timezone starting near midnight, but not confirmed.
4. **Most promising features:** V17, V14, V12, V10, V16, V3, V7, V11, note: I will also check Amount and Time - they don't have strong correlation, but they're worth testing in non-linear models
5. **Model strategy:** PR-AUC as main metric (better on imbalanced data, unlike ROC-AUC), F1, Precision, Recall tracked